<h1>Astrometry: finding a planet from the wobble of its star on the sky</h1>

A planet and its star both orbit their common centre of mass. Remember that the radial velocity (RV) method measures the star's motion along our line of sight (i.e. velocity in the radial direction). **Astrometry** measures the star's motion across the sky: over one orbit, the star traces a tiny ellipse on the plane of the sky. For a planet of mass $M_{\rm p}$ on an orbit with semi-major axis $a$ around a star of mass $M_*$ at distance $d$, the size of the star's ellipse (its *astrometric signature*) is

$$ \alpha \approx \frac{M_{\rm p}}{M_*}\,\frac{a}{d} $$

where $\alpha$ is in arcseconds for $a$ in au and $d$ in parsecs (and assuming $M_{\rm p} \ll M_*$). Jupiter moves the Sun by $\alpha \approx 0.5$ milliarcseconds (mas) as seen from 10 parsecs, which is about the angle covered by a one-metre object on the Moon, as seen from Earth.

Between 2014 and 2025, the European Space Agency's **Gaia** satellite measured the positions of almost two billion stars, many times each. Each time a star crosses Gaia's field of view, Gaia measures its position 8 or 9 times, but only along the direction in which it is scanning the sky at that moment; this direction changes from one observation to the next. ESA has released these individual measurements early (ahead of Gaia's fourth data release, due in December 2026) for a few stars, including **Gaia-4**, which has a giant planet, **Gaia-4 b**, which was discovered by Gaia. This notebook first simulates Gaia's measurements of a star with a planet (part 1), and then analyses the real measurements of Gaia-4 (part 2).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from html import escape
from IPython.display import display, HTML
import ipywidgets as widgets

import astrometry_tools as at

# Figures shown with widgets live in their own output area (widgets.Output).
# show_in() replaces what an output area shows, sending the new content to the browser once.
# (Using 'with out: display(...)' instead sends it back and forth, and at the top level of a
# cell it can make Voila, the app view, hang.)
def show_in(out, obj):
    with out.hold_sync():
        out.outputs = ()
        out.append_display_data(obj)

def busy(out, text='Working, please wait...'):
    show_in(out, HTML(f'<p><i>{text}</i></p>'))

# Try and make plots look nice
plt.rcParams["font.family"] = 'serif'
plt.rcParams["mathtext.fontset"] = 'stix'
plt.rcParams["font.serif"] = ['Palatino', 'TeX Gyre Pagella', 'DejaVu Serif']   # first available is used
plt.rcParams['pdf.fonttype'] = 42

style = {'description_width': 'initial'}
try:
    S                                  # results of each step of part 2, passed on to the next step
except NameError:                      # (kept if this cell is run again)
    S = {}


def plus_minus(value, error):
    # value and uncertainty with the same number of decimals (2 significant figures in the uncertainty)
    if not np.isfinite(error) or error <= 0:
        return f'{value:.4g}'
    decimals = max(0, 1 - int(np.floor(np.log10(error))))
    return f'{value:.{decimals}f} &plusmn; {error:.{decimals}f}'

<h2>Part 1: simulating Gaia's measurements</h2>

We simulate a star of one solar mass with a planet, observed by Gaia at the same times and in the same scanning directions as Gaia-4 (part 2), and with the same measurement uncertainties. Use the sliders to change the planet's mass (above about 13 Jupiter masses it would be a brown dwarf) and orbit (inclination 0: orbit seen face-on; 90: edge-on), and the distance of the star.

The four panels show:
- **(a)** the star's path on the sky during the 5 years of Gaia data used here: mostly its steady motion across the sky (its *proper motion*);
- **(b)** the same as (a) but without the proper motion, which reveals the yearly parallax ellipse (Gaia orbits the Sun along with Earth), plus the wobble caused by the planet;
- **(c)** the wobble alone: the star's orbit around the centre of mass, with its positions at Gaia's observations (dots), and the size of a typical measurement uncertainty (bar);
- **(d)** what is left when we fit the motion of a star *without* a planet (its position, parallax and proper motion) to the simulated measurements, averaged per observation. If the planet can be detected, a pattern remains, and the reduced $\chi^2$ of this fit is clearly above 1.

Things to try: (i) simulate Jupiter (1 Jupiter mass, period of 4333 days) at 10 pc, and then further away; (ii) a planet like Gaia-4 b (12 Jupiter masses, period of 570 days, at 74 pc (note that the simulated star has a higher mass than Gaia-4, $1~\mathrm{M}_{\odot}$ instead of $0.64~\mathrm{M}_{\odot}$, so its wobble is smaller: about 0.2 instead of 0.3 mas); and (iii) a planet with a period much longer than 5 years (over 5 years its wobble then looks almost like a straight-line motion, so most of it is absorbed into the proper motion).

In [ ]:
TEMPLATE = at.load('Gaia-4')          # times, scanning directions, parallax factors and uncertainties of Gaia-4
PM = (-60.0, 20.0)                     # proper motion of the simulated star (mas/yr)

w_mp = widgets.FloatLogSlider(value=12, base=10, min=-0.5, max=np.log10(80), step=0.01, description='Planet mass (Jupiter masses):',
                              readout_format='.3r', continuous_update=False, style=style, layout=widgets.Layout(width='520px'))
w_P = widgets.FloatLogSlider(value=570, base=10, min=1, max=4, step=0.01, description='Period (days):',
                             readout_format='.4r', continuous_update=False, style=style, layout=widgets.Layout(width='520px'))
w_e = widgets.FloatSlider(value=0.3, min=0, max=0.9, step=0.01, description='Eccentricity:', continuous_update=False, style=style)
w_inc = widgets.FloatSlider(value=60, min=0, max=90, step=1, description='Inclination (degrees):', continuous_update=False,
                            style=style)
w_dist = widgets.FloatLogSlider(value=74, base=10, min=0.5, max=3, step=0.01, description='Distance (pc):',
                                readout_format='.3r', continuous_update=False, style=style, layout=widgets.Layout(width='520px'))
out_sim = widgets.Output()
display(w_mp, w_P, w_e, w_inc, w_dist, out_sim)


def simulate(change=None):
    d = TEMPLATE
    sim = at.simulate(d, w_mp.value*at.MJUP, 1.0, w_P.value, w_e.value, np.radians(w_inc.value), w_dist.value, pm=PM)
    p = sim['params']
    fit = at.single_star_fit(dict(d, w=sim['w']))
    tt = np.linspace(d['t'].min(), d['t'].max(), 3000)
    track = dict(plx=sim['plx'], pm=PM, A=p['A'], B=p['B'], F=p['F'], G=p['G'], period=sim['period'], e=sim['e'], M0=sim['M0'])
    x, y = at.sky_track(tt, d['ra0'], d['dec0'], **track)
    xo, yo = at.sky_track(d['t'], d['ra0'], d['dec0'], **track)
    E = np.linspace(0, 2*np.pi, 400)                      # the whole orbit, as a smooth curve
    X, Y = np.cos(E) - sim['e'], np.sqrt(1 - sim['e']**2)*np.sin(E)
    Xo, Yo = at.orbit_xy(d['t'], sim['period'], sim['e'], sim['M0'])
    sigma = np.median(d['sigma'])

    fig, axes = plt.subplots(2, 2, figsize=(10, 9))
    plt.close(fig)
    (ax1, ax2), (ax3, ax4) = axes
    ax1.plot(x, y, '-', color='k', lw=1)
    ax1.plot(xo, yo, '.', color='tab:blue', ms=3)
    ax1.set_title('(a) path on the sky')
    ax2.plot(x - PM[0]*tt, y - PM[1]*tt, '-', color='k', lw=0.7)
    ax2.plot(xo - PM[0]*d['t'], yo - PM[1]*d['t'], '.', color='tab:blue', ms=3)
    ax2.set_title('(b) proper motion removed: parallax + wobble')
    ax3.plot(p['B']*X + p['G']*Y, p['A']*X + p['F']*Y, '-', color='k', lw=1)
    ax3.plot(p['B']*Xo + p['G']*Yo, p['A']*Xo + p['F']*Yo, '.', color='tab:blue', ms=4)
    ax3.plot(0, 0, '+', color='red', ms=10)
    xorb, yorb = p['B']*X + p['G']*Y, p['A']*X + p['F']*Y
    lim = max(1.15*np.max(np.abs(np.r_[xorb, yorb])), 1.2*sigma)
    ax3.set_xlim(lim, -lim)
    ax3.set_ylim(-1.45*lim, lim)                          # room below the orbit for the uncertainty bar
    ax3.errorbar([0], [-1.3*lim], xerr=[sigma/2], color='tab:red', capsize=3)
    ax3.text(0, -1.22*lim, f'uncertainty of one measurement ({sigma:.2f} mas)', color='tab:red', ha='center', fontsize=9)
    ax3.set_title(f'(c) wobble: $\\alpha$ = {sim["a0"]:.3g} mas')
    for ax in (ax1, ax2):
        ax.invert_xaxis()
    for ax in (ax1, ax2):
        ax.set_aspect('equal', adjustable='datalim')
    ax3.set_aspect('equal', adjustable='box')
    for ax in (ax1, ax2, ax3):
        ax.set_xlabel('R.A. offset (mas, east to the left)')
        ax.set_ylabel('Dec. offset (mas)')
    tm, rm, em = at.transit_means(d, fit['resid'])
    ax4.errorbar(2017.5 + tm, rm, yerr=em, fmt='o', ms=3, color='tab:blue', elinewidth=0.8)
    ax4.axhline(0, color='k', lw=0.8)
    ax4.set_xlabel('Year')
    ax4.set_ylabel('Measured minus model without planet (mas)')
    ax4.set_title(f'(d) fit of a star without a planet: reduced $\\chi^2$ = {fit["chi2_red"]:.2f}')
    fig.tight_layout()
    a_au, a_star = sim['a'], sim['a1']
    show_in(out_sim, fig)
    out_sim.append_display_data(HTML(
        f'<p>Orbit of the planet: a = {a_au:.3g} au; the star orbits the centre of mass at a<sub>*</sub> = {a_star:.3g} au, which at {w_dist.value:.4g} pc '
        f'is α = {sim["a0"]:.3g} mas, {sim["a0"]/sigma:.2g} times the uncertainty of one measurement '
        f'({sigma:.2f} mas). Parallax {sim["plx"]:.3g} mas.</p>'))


for w in (w_mp, w_P, w_e, w_inc, w_dist):
    w.observe(simulate, names='value')
simulate()

<h2>Part 2: the real measurements of Gaia-4</h2>

**Gaia-4** is a cool star of 0.64 solar masses, 74 pc away. Gaia's third data release (2022) included an orbit for this star, with a period of about 570 days, and radial-velocity measurements confirmed the cause: a giant planet of 11.8 Jupiter masses, **Gaia-4 b** (Stefánsson et al. 2025). The star's wobble is tiny: about 0.3 mas, only about three times the uncertainty of a single measurement. The measurements used here are those used by Gaia's own analysis (Gaia DR4 pre-release).

<h3>1. The measurements</h3>

The top panel shows the measured positions $w$ (along the scanning direction): they are dominated by the star's proper motion and parallax, seen along the changing scanning directions (middle panel: the *scan angle*). The bottom panel shows the *parallax factor*: how much of the star's parallax shift lies along the scanning direction at that moment.

In [ ]:
out_data = widgets.Output()
display(out_data)


def show_data():
    d = S['d']
    span = d['t'].max() - d['t'].min()
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 7.5), sharex=True, gridspec_kw=dict(height_ratios=[2, 1, 1]))
    plt.close(fig)
    year = 2017.5 + d['t']
    ax1.plot(year, d['w'], '.', ms=2, color='k')
    ax1.set_ylabel('$w$ (mas)')
    ax1.set_title(f"Gaia-4: {len(d['w'])} measurements in {len(np.unique(d['transit']))} observations over {span:.1f} years")
    ax2.plot(year, np.degrees(d['theta']), '.', ms=2, color='tab:blue')
    ax2.set_ylabel(r'scan angle $\theta$ (deg)')
    ax3.plot(year, d['f'], '.', ms=2, color='tab:green')
    ax3.set_ylabel('parallax factor')
    ax3.set_xlabel('Year')
    fig.tight_layout()
    show_in(out_data, fig)


if 'd' in S:                           # (if this cell is run again)
    show_data()

<h3>2. A star without a planet</h3>

We first fit the motion of a star *without* a planet: its position (R.A. and Dec), parallax and proper motion (in both R.A. and Dec directions, i.e. 5 numbers in total), using least squares. The plot shows what is left: the measurements minus this model, averaged per observation set. For Gaia-4 the reduced $\chi^2$ is about 3.5 instead of about 1, which tells us that there is more than just a star and noise in the data, although the pattern is hard to see by eye. (From here on, one measurement that lies far from the others, more than 5 times its uncertainty away from the best orbit found in step 4, is left out.)

In [ ]:
out_single = widgets.Output()
display(out_single)


def single_star():
    d = S['d']
    dk = at.subset(d, S['keep'])
    fit = S['single'] = at.single_star_fit(dk)
    tm, rm, em = at.transit_means(dk, fit['resid'])
    fig, ax = plt.subplots(figsize=(10, 3.8))
    plt.close(fig)
    ax.errorbar(2017.5 + tm, rm, yerr=em, fmt='o', ms=3, color='tab:blue', elinewidth=0.8)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xlabel('Year')
    ax.set_ylabel('Measured minus model (mas)')
    ax.set_title(f"Gaia-4, star without a planet: reduced $\\chi^2$ = {fit['chi2_red']:.2f}")
    fig.tight_layout()
    show_in(out_single, fig)


if 'd' in S:                           # (if this cell is run again)
    single_star()

<h3>3. Searching for the period</h3>

We now add an orbit to the model, testing many trial periods, and see how much better the fit gets (i.e. how much $\chi^2$ decreases): how much injecting a given period improves the fit is shown in the **periodogram**. A planet shows up as a peak at its orbital period; the dashed line shows the published period. (Some smaller peaks come from the way Gaia's scanning sampled the star, rather than from the star itself.)

In [ ]:
out_pgram = widgets.Output()
display(out_pgram)


def search():
    d = S['d']
    periods, delta = at.periodogram(at.subset(d, S['keep']))
    S['peak'] = periods[np.argmax(delta)]
    fig, ax = plt.subplots(figsize=(10, 3.8))
    plt.close(fig)
    ax.plot(periods, delta, '-', color='k', lw=0.8)
    ax.axvline(at.SYSTEMS['Gaia-4']['published']['period'], color='tab:red', ls='--', lw=1, label='published period')
    ax.set_xscale('log')
    ax.set_xlabel('Trial period (days)')
    ax.set_ylabel(r'Improvement in $\chi^2$')
    ax.set_title(f"Gaia-4: highest peak at {S['peak']:.4g} days")
    ax.legend(loc='upper left')
    fig.tight_layout()
    show_in(out_pgram, fig)


if 'd' in S:                           # (if this cell is run again)
    search()

<h3>4. Fitting the orbit</h3>

The star's orbit is described by its period $P$, its eccentricity $e$, where the star is in its orbit at a reference time (the *mean anomaly* at 2017.5, an angle that grows by 360 degrees every period), and the size and orientation of the orbit on the sky. Set the first three with the sliders: the size and orientation of the orbit (and the star's position, parallax and proper motion) are then fitted instantly. **Best fit** searches for the best values of all of them, starting from the highest peak of the periodogram. The sliders start at that peak, for a circular orbit.

The left plot shows the star's motion along the scanning direction caused by the planet (the measurements minus the motion of a star without a planet, averaged per observation, in blue) and the model at the same times (red); below it is what is left after subtracting the full model. The measurement that is left out is shown as a red cross. The right plot shows the fitted orbit of the star on the sky.

In the table, $\alpha$ is the size of the star's orbit (its angular semi-major axis, in mas, as in part 1) and $i$ its inclination (0: seen face-on; 90 degrees: edge-on; an inclination $i$ above 90 degrees looks like $180 - i$, but with the star going round its orbit the other way). The reduced $\chi^2$ stays a little above 1 (the uncertainties of single measurements are a little too small), so the uncertainties in the table are scaled up by $\sqrt{\chi^2_{\rm red}}$. While you move the sliders, $P$, $e$ and the mean anomaly are set by hand rather than fitted, and the other uncertainties assume they are exact; press **Best fit** for the full uncertainties.

In [ ]:
w_period = widgets.FloatLogSlider(value=500, base=10, min=1, max=np.log10(6000), step=0.001, description='Period (days):',
                                  readout_format='.5g', continuous_update=False, style=style, layout=widgets.Layout(width='600px'))
w_ecc = widgets.FloatSlider(value=0, min=0, max=0.95, step=0.005, description='Eccentricity:', readout_format='.3f',
                            continuous_update=False, style=style, layout=widgets.Layout(width='600px'))
w_phase = widgets.FloatSlider(value=0, min=0, max=360, step=0.5, description='Mean anomaly at 2017.5 (deg):',
                              continuous_update=False, style=style, layout=widgets.Layout(width='600px'))
w_best = widgets.Button(description='Best fit')
out_orbit = widgets.Output()
display(w_period, w_ecc, w_phase, w_best, out_orbit)
ROWS = [('period', 'period P (days)'), ('e', 'eccentricity e'), ('a0', 'size of the star\'s orbit α (mas)'),
        ('inc', 'inclination i (deg)'), ('plx', 'parallax ϖ (mas)')]


def orbit(change=None):
    if S.get('quiet') or 'd' not in S:
        return
    try:
        d = S['d']
        dk = at.subset(d, S['keep'])
        fit = S['orbit'] = at.orbit_fit(dk, w_period.value, w_ecc.value, np.radians(w_phase.value))
        best = S.get('best')
        full = S['full'] = best is not None and np.allclose([w_period.value, w_ecc.value, w_phase.value], best, rtol=1e-6, atol=1e-6)
        info = at.SYSTEMS['Gaia-4']
        try:
            u = at.uncertainties(dk, fit, info['host_mass'], info['host_mass_err'], full=full)
        except Exception:
            u = None                           # draw the orbit anyway, without uncertainties
        S['unc'] = u
        draw_orbit(d, S['keep'], fit, u)
    except Exception as error:
        S['unc'] = None
        show_in(out_orbit, HTML(f'<p style="color:red">{escape(str(error))}</p>'))
    if 'planet_mass' in globals():
        planet_mass()


def draw_orbit(d, keep, fit, u):
    p = fit['params']
    dk = at.subset(d, keep)
    tm, om, em = at.transit_means(dk, dk['w'] - fit['single_star_part'])
    _, mm, _ = at.transit_means(dk, fit['orbit_part'])
    _, rm, _ = at.transit_means(dk, fit['resid'])
    out = at.subset(d, ~keep)                  # the measurement left out: its position relative to the model
    D = at.design(out, fit['period'], fit['e'], fit['M0'])
    coef = np.array([p[k] for k in at.LINEAR])
    fig = plt.figure(figsize=(11, 5.5))
    plt.close(fig)
    ax1 = fig.add_axes([0.07, 0.36, 0.5, 0.56])
    ax2 = fig.add_axes([0.07, 0.1, 0.5, 0.22], sharex=ax1)
    ax3 = fig.add_axes([0.66, 0.1, 0.32, 0.82])
    ax1.errorbar(2017.5 + tm, om, yerr=em, fmt='o', ms=3, color='tab:blue', elinewidth=0.8, label='measured (observation averages)')
    ax1.plot(2017.5 + tm, mm, 'o', ms=3, mfc='none', color='tab:red', label='model')
    if len(out['t']):
        ax1.plot(2017.5 + out['t'], out['w'] - D[:, :5] @ coef[:5], 'x', ms=5, color='tab:red', label=f"left out ({len(out['t'])})")
    ax1.set_ylabel('Motion along scan due to the planet (mas)')
    ax1.set_title(f"Gaia-4: reduced $\\chi^2$ = {fit['chi2_red']:.2f}")
    ax1.legend(loc='upper left', fontsize=8)
    plt.setp(ax1.get_xticklabels(), visible=False)
    ax2.errorbar(2017.5 + tm, rm, yerr=em, fmt='o', ms=3, color='tab:blue', elinewidth=0.8)
    ax2.axhline(0, color='k', lw=0.8)
    lim2 = 1.3*np.max(np.abs(rm) + em)
    ax2.set_ylim(-lim2, lim2)
    ax2.set_xlabel('Year')
    ax2.set_ylabel('Residual (mas)')
    E = np.linspace(0, 2*np.pi, 400)
    X, Y = np.cos(E) - fit['e'], np.sqrt(1 - fit['e']**2)*np.sin(E)
    x, y = p['B']*X + p['G']*Y, p['A']*X + p['F']*Y
    ax3.plot(x, y, '-', color='tab:red')
    ax3.plot(x[0], y[0], 'o', color='tab:red', ms=6, label='periastron')
    k = 40                                     # direction of motion
    ax3.annotate('', xy=(x[k + 5], y[k + 5]), xytext=(x[k], y[k]), arrowprops=dict(arrowstyle='->', color='tab:red'))
    ax3.plot(0, 0, '+', color='k', ms=10, label='centre of mass')
    lim = 1.15*np.max(np.abs(np.r_[x, y]))
    ax3.set_xlim(lim, -lim)
    ax3.set_ylim(-lim, lim)
    ax3.set_aspect('equal')
    ax3.set_xlabel('R.A. offset (mas, east to the left)')
    ax3.set_ylabel('Dec. offset (mas)')
    ax3.set_title('Orbit of the star on the sky')
    ax3.legend(loc='lower left', fontsize=8)
    show_in(out_orbit, fig)
    out_orbit.append_display_data(HTML(results_table(fit, u)))


def results_table(fit, u):
    pub = at.SYSTEMS['Gaia-4']['published']
    rows = []
    for key, label in ROWS:
        if u is None:
            mine = ''
        else:
            val, err = u[key]
            mine = f'{val:.6g} (set by slider)' if not np.isfinite(err) else plus_minus(val, err)
        ps = '' if key not in pub else f"{pub[key]:g}" + (f" &plusmn; {pub[key + '_err']:g}" if key + '_err' in pub else '')
        if key == 'plx':
            ps = '13.643 &plusmn; 0.025 (Gaia DR3 orbit solution)'
        rows.append(f'<tr><td>{label}</td><td>{mine}</td><td>{ps}</td></tr>')
    note = f"Reduced χ² {fit['chi2_red']:.2f} (star without a planet: {S['single']['chi2_red']:.2f})."
    if not S.get('full'):
        note += ' Period, eccentricity and mean anomaly set by the sliders: press <b>Best fit</b> for the full uncertainties.'
    return ('<table><tr><th></th><th>this fit</th><th>published (Stefánsson et al. 2025)</th></tr>' + ''.join(rows) +
            f'</table><p>{note}</p>')


def sliders_to(P, e, M0_deg):
    S['quiet'] = True
    try:
        w_period.value, w_ecc.value, w_phase.value = float(P), float(e), float(np.mod(M0_deg, 360))
    finally:
        S['quiet'] = False


def best_fit(button=None):
    if 'd' not in S:
        show_in(out_orbit, HTML('<p style="color:red">Run the last code cell (Run part 2) first.</p>'))
        return
    busy(out_orbit, 'Searching for the best orbit...')
    try:
        fit = at.best_orbit(at.subset(S['d'], S['keep']))
        sliders_to(fit['period'], fit['e'], np.degrees(fit['M0']))
        S['best'] = (w_period.value, w_ecc.value, w_phase.value)
        orbit()
    except Exception as error:
        show_in(out_orbit, HTML(f'<p style="color:red">{escape(str(error))}</p>'))


for w in (w_period, w_ecc, w_phase):
    w.observe(orbit, names='value')
w_best.on_click(best_fit)
if 'orbit' in S:                               # (if this cell is run again)
    sliders_to(S['orbit']['period'], S['orbit']['e'], np.degrees(S['orbit']['M0']))
    orbit()

<h3>5. The mass of the planet</h3>

The star's orbit around the centre of mass has semi-major axis $a_* = \alpha/\varpi$ (in au, with $\alpha$ (as defined in part 1) and the parallax $\varpi$ in mas). Kepler's third law then gives

$$\frac{(\alpha/\varpi)^3}{(P/{\rm yr})^2} ~=~ \frac{M_{\rm p}^3}{(M_* + M_{\rm p})^2} ~\approx~ \frac{M_{\rm p}^3}{M_*^2} \qquad (\text{masses in solar masses}),$$

so the planet's mass $M_{\rm p}$ follows once the star's mass $M_*$ is known (from its spectrum; you can change it). Because Gaia measures the inclination of the orbit, this is the planet's true mass, not the minimum mass $M_{\rm p}\sin i$ that the radial velocity method gives.

In [ ]:
w_mstar = widgets.BoundedFloatText(value=at.SYSTEMS['Gaia-4']['host_mass'], min=0.05, max=5, step=0.001,
                                    description='Mass of the star M* (solar masses):', style=style,
                                    layout=widgets.Layout(width='340px'))
out_mass = widgets.Output()
display(w_mstar, out_mass)


def planet_mass(change=None):
    if S.get('quiet') or 'd' not in S:
        return
    u = S.get('unc')
    if u is None:
        show_in(out_mass, HTML('<p><i>No mass: the orbit above could not be evaluated.</i></p>'))
        return
    try:
        info = at.SYSTEMS['Gaia-4']
        pub = info['published']
        fM = u['fM'][0]
        mp = at.companion_mass(fM, w_mstar.value)
        if not np.isfinite(mp):
            show_in(out_mass, HTML('<p>No physical mass for this orbit (the left-hand side above must be positive).</p>'))
            return
        err = u['m2'][1]/u['m2'][0]*mp if (np.isclose(w_mstar.value, info['host_mass']) and np.isfinite(u['m2'][1])) else np.nan
        mp_j = mp/at.MJUP
        kind = ('This is below about 13 Jupiter masses (where deuterium fusion would start: the usual boundary between '
                'planets and brown dwarfs), so Gaia-4 b is a <b>planet</b>.' if mp_j < 13 else
                'This is above about 13 Jupiter masses (where deuterium fusion would start: the usual boundary between '
                'planets and brown dwarfs): for this mass of the star, Gaia-4 b would be a <b>brown dwarf</b>.')
        mstar = f'{w_mstar.value:.3g}' + ('.0' if w_mstar.value == int(w_mstar.value) else '')
        show_in(out_mass, HTML(
            f"<p>For a star of M<sub>*</sub> = {mstar} solar masses, the planet has "
            f"<b>M<sub>p</sub> = {plus_minus(mp/at.MJUP, err/at.MJUP)} Jupiter masses</b> "
            f"(published: {pub['m2']/at.MJUP:.3g} &plusmn; {pub['m2_err']/at.MJUP:.2g} Jupiter masses, for M<sub>*</sub> = "
            f"{info['host_mass']} &plusmn; {info['host_mass_err']} solar masses).</p>"
            + ('<p><i>(Uncertainty shown only for the published mass of the star.)</i></p>' if not np.isfinite(err) else
               '' if S.get('full') else '<p><i>(Period, eccentricity and mean anomaly set by the sliders: this uncertainty assumes '
               'they are exact. Press Best fit for the full uncertainty.)</i></p>')
            + f'<p>{kind}</p>'))
    except Exception as error:
        show_in(out_mass, HTML(f'<p style="color:red">{escape(str(error))}</p>'))


w_mstar.observe(planet_mass, names='value')
if 'unc' in S:                         # (if this cell is run again)
    planet_mass()

In [ ]:
# Run part 2
STEPS = ['data', 'single', 'search', 'orbit']


def run_all():
    outs = dict(data=out_data, single=out_single, search=out_pgram, orbit=out_orbit)
    for s in STEPS:
        busy(outs[s])
    busy(out_mass, '...')
    for i, s in enumerate(STEPS):
        try:
            if s == 'data':
                S.clear()
                d = at.load('Gaia-4')
                S['keep'] = at.best_orbit(d, clip=5.0)['keep']          # the outlying measurement(s), left out from step 2 on
                S['d'] = d
                show_data()
            elif s == 'single':
                single_star()
            elif s == 'search':
                search()
            elif s == 'orbit':
                S['best'] = None
                sliders_to(S['peak'], 0.0, 0.0)
                orbit()
        except Exception as error:
            show_in(outs[s], HTML(f'<p style="color:red">This step failed: {escape(str(error))}</p>'))
            for out in [outs[s2] for s2 in STEPS[i + 1:]] + [out_mass]:
                show_in(out, HTML('<p><i>Not calculated, because an earlier step failed.</i></p>'))
            return


run_all()

<h3>References</h3>

Gaia Collaboration, Prusti et al. 2016, A&A 595, A1 (the Gaia mission) $\cdot$ Gaia Collaboration, Vallenari et al. 2023, A&A 674, A1 (Gaia DR3) $\cdot$ Holl et al. 2023, A&A 674, A10 (Gaia astrometric orbits) $\cdot$ Lindegren et al. 2021, A&A 649, A2 (Gaia astrometric solution) $\cdot$ Stefánsson et al. 2025, AJ 169, 107 (Gaia-4 b).

Data: Gaia DR4 pre-release epoch astrometry (https://www.cosmos.esa.int/web/gaia/dr4-prerelease), ESA/Gaia/DPAC, licensed under CC BY-NC 3.0 IGO (https://creativecommons.org/licenses/by-nc/3.0/igo/); selected and reformatted for this notebook. This work has made use of data from the European Space Agency (ESA) mission Gaia (https://www.cosmos.esa.int/gaia), processed by the Gaia Data Processing and Analysis Consortium (DPAC, https://www.cosmos.esa.int/web/gaia/dpac/consortium). Funding for the DPAC has been provided by national institutions, in particular the institutions participating in the Gaia Multilateral Agreement.